# 35 — Automation: Packer, Ansible & Scripting

**Role:** Senior AWS DevOps Engineer | **Focus:** AMI Pipelines, Configuration Management, Bash/Python/PowerShell

---

## Section A: Packer & AMI Management

### Q1: Golden AMI Pipeline
**Question:** Describe your process for building and maintaining a \"Golden AMI\" that all EC2 instances and EKS nodes use.

**Expected Answer:**
- **Base image**: Start from official Amazon Linux 2023 or Ubuntu AMI.
- **Packer template**: HCL2 format. Source → Provisioners (shell, Ansible) → Post-processors.
- **Hardening**: CIS benchmark compliance, remove unnecessary packages, configure SSH, install CloudWatch agent.
- **Pipeline**: GitLab CI triggers Packer build weekly or on commit. Test AMI (launch, run Inspec/Goss tests). Promote to shared AMI.
- **Versioning**: Tag AMIs with build date, commit SHA. Clean up AMIs older than 90 days.
- **EKS**: Custom AMIs for EKS nodes must include `amazon-eks-node` bootstrap script.

```hcl
# Packer HCL2 example
source \"amazon-ebs\" \"golden\" {
  ami_name      = \"golden-{{timestamp}}\"
  instance_type = \"t3.medium\"
  region        = \"us-east-1\"
  source_ami_filter {
    filters = {
      name = \"al2023-ami-*-x86_64\"
    }
    owners      = [\"amazon\"]
    most_recent = true
  }
  ssh_username = \"ec2-user\"
}

build {
  sources = [\"source.amazon-ebs.golden\"]
  provisioner \"ansible\" {
    playbook_file = \"./playbooks/harden.yml\"
  }
}
```

---

### Q2: AMI Patching Strategy
**Question:** How do you ensure all running EC2 instances are using the latest patched AMI without manual intervention?

**Expected Answer:**
- **Immutable infrastructure**: Don't patch running instances. Build new AMI → rolling replacement.
- **ASG**: Update Launch Template with new AMI ID → Instance Refresh (rolling update).
- **EKS**: Update node group AMI → managed node group handles rolling replacement.
- **Terraform**: Update `ami` in the launch template, `apply` triggers instance refresh.
- **Automation**: Weekly Packer build → Terraform auto-updates AMI var → MR created for review.
- **Compliance**: AWS Systems Manager Patch Manager for tracking patch compliance.

## Section B: Ansible Configuration Management

### Q3: Ansible vs. Baking into AMI/Docker
**Question:** When do you use Ansible at runtime (on instance boot) vs. baking everything into the AMI or Docker image?

**Expected Answer:**
- **Bake (AMI/Docker)**: Faster boot, deterministic, no external dependencies at startup. Preferred for containers and immutable infra.
- **Runtime Ansible**: Needed for environment-specific config (secrets, endpoints), dynamic inventory, or legacy systems that can't be containerized.
- **Hybrid**: Bake 90% into AMI (packages, hardening). Use Ansible/User Data for the last 10% (register with monitoring, fetch runtime secrets).
- **Containers**: Almost everything baked into Docker image. Runtime config via env vars and mounted secrets.

---

### Q4: Ansible Best Practices
**Question:** How do you structure a large Ansible codebase for managing both Linux and Windows servers?

**Expected Answer:**
- **Roles**: Reusable roles per concern (`common`, `hardening`, `monitoring`, `web-server`).
- **Inventory**: Dynamic inventory plugin for AWS (ec2.py or aws_ec2 plugin). Group by tags.
- **Variables**: `group_vars/linux.yml`, `group_vars/windows.yml` for OS-specific settings.
- **Windows**: Use `win_*` modules. Connection via WinRM (not SSH). `ansible_connection: winrm`.
- **Idempotency**: Every task should be safe to run multiple times without side effects.
- **Testing**: Molecule for role testing. Lint with `ansible-lint`.
- **Vault**: `ansible-vault` for encrypting sensitive variables in the repo.

## Section C: Scripting Challenges

### Q5: Bash — Log Analysis
**Question:** Write a Bash one-liner that finds the top 10 IP addresses by request count from an Nginx access log.

**Expected Answer:**
```bash
awk '{print $1}' /var/log/nginx/access.log | sort | uniq -c | sort -rn | head -10
```
**Follow-up:** How would you alert if any IP exceeds 10,000 requests in 5 minutes?
- Use `fail2ban`, or a cron job piping to CloudWatch custom metric, or WAF rate-based rules.

---

### Q6: Python — AWS Resource Cleanup
**Question:** Write a Python script (boto3) to find and delete all unattached EBS volumes older than 30 days.

**Expected Answer:**
```python
import boto3
from datetime import datetime, timezone, timedelta

ec2 = boto3.client('ec2')
cutoff = datetime.now(timezone.utc) - timedelta(days=30)

volumes = ec2.describe_volumes(
    Filters=[{'Name': 'status', 'Values': ['available']}]
)['Volumes']

for vol in volumes:
    if vol['CreateTime'] < cutoff:
        print(f"Deleting {vol['VolumeId']} created {vol['CreateTime']}")
        # ec2.delete_volume(VolumeId=vol['VolumeId'])  # Uncomment to execute
```
**Follow-up:** How would you make this safe? (DRY-RUN mode, require tag `DoNotDelete` check, SNS notification before deletion.)

---

### Q7: PowerShell — Windows Server Management
**Question:** Write a PowerShell script that checks disk space on all drives and sends an alert if any drive is over 90% full.

**Expected Answer:**
```powershell
$drives = Get-WmiObject Win32_LogicalDisk -Filter \"DriveType=3\"
foreach ($drive in $drives) {
    $usedPercent = [math]::Round(($drive.Size - $drive.FreeSpace) / $drive.Size * 100, 2)
    if ($usedPercent -gt 90) {
        Write-Warning \"$($drive.DeviceID) is $usedPercent% full!\"
        # Send to CloudWatch or SNS
        aws cloudwatch put-metric-data `
            --namespace \"Custom/DiskSpace\" `
            --metric-name \"DiskUsedPercent\" `
            --dimensions \"Drive=$($drive.DeviceID)\" `
            --value $usedPercent
    }
}
```

---

### Q8: Cross-Platform Scripting
**Question:** You manage both Linux and Windows environments. How do you write automation that works across both?

**Expected Answer:**
- **Python**: Cross-platform. Use `boto3`, `paramiko`, `pywinrm`. Best for complex logic.
- **Ansible**: Abstracts OS differences via modules (`package` vs `win_package`).
- **AWS SSM Run Command**: Execute scripts on any managed instance. Supports Bash AND PowerShell documents.
- **Avoid**: Bash-only scripts for mixed environments. PowerShell Core (pwsh) runs on Linux but adoption is low.
- **Testing**: Use CI matrix to test scripts on both Amazon Linux and Windows Server.